## Model Improvement

### Cross Vaidation

In [1]:
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Small sample dataset
np.random.seed(1)
X = np.random.randn(50, 2)
y = (X[:,0] + X[:,1] > 0).astype(int)

model = LogisticRegression()

# First, the manual approach, to see what happens inside
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_num = 1
for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f"Fold {fold_num}: Test size={len(test_idx)}, Accuracy={acc:.2%}")
    fold_num += 1

print()
# Shortcut: everything in one line
scores = cross_val_score(LogisticRegression(), X, y, cv=5)
print(f"Using cross_val_score directly: {scores.round(2)}")
print(f"Mean Accuracy: {scores.mean():.2%}")

Fold 1: Test size=10, Accuracy=80.00%
Fold 2: Test size=10, Accuracy=100.00%
Fold 3: Test size=10, Accuracy=100.00%
Fold 4: Test size=10, Accuracy=90.00%
Fold 5: Test size=10, Accuracy=100.00%

Using cross_val_score directly: [1.  1.  1.  0.9 0.8]
Mean Accuracy: 94.00%


#### Stratified KFold

In [2]:
from sklearn.model_selection import KFold, StratifiedKFold

kf = KFold(n_splits=5)                # ignores class balance
skf = StratifiedKFold(n_splits=5)     # maintains class balance

In [3]:
# STEP 1: COMPARE models with cross-validation to decide which to use
from sklearn.model_selection import cross_val_score

model_A = LogisticRegression()
model_B = RandomForestClassifier()

scores_A = cross_val_score(model_A, X_train, y_train, cv=5)
scores_B = cross_val_score(model_B, X_train, y_train, cv=5)

print(f"Logistic Regression: {scores_A.mean():.2%}")
print(f"Random Forest: {scores_B.mean():.2%}")
# Socho Random Forest better nikla

# STEP 2: train the FINAL model on ALL the training data (no cross-validation now)
final_model = RandomForestClassifier()
final_model.fit(X_train, y_train)   # <-- this is the "actual, deployable" model

# STEP 3: Final, honest evaluation on the unseen test set
final_accuracy = final_model.score(X_test, y_test)
print(f"Final Test Accuracy: {final_accuracy:.2%}")

Logistic Regression: 92.50%
Random Forest: 85.00%
Final Test Accuracy: 100.00%


### Grid Search CV

In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Step 1: Split (ek hi baar)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: GridSearchCV: try hyperparameters and evaluate them with CV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None]
}

grid_search = GridSearchCV(
    RandomForestClassifier(), 
    param_grid, 
    cv=5,                    # <-- CV is specified here
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)   # <-- ONLY on the training data! The test data has not been touched yet

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.2%}")

# Step 3: the best model is already trained (automatically) on ALL of X_train
final_model = grid_search.best_estimator_

# Step 4: use the test set for the FIRST time: final, honest evaluation
test_accuracy = final_model.score(X_test, y_test)
print(f"Final Test Accuracy: {test_accuracy:.2%}")

Best params: {'max_depth': 7, 'n_estimators': 100}
Best CV score: 90.00%
Final Test Accuracy: 90.00%


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import time

np.random.seed(42)
n_samples = 300
time_spent = np.random.normal(12, 5, n_samples).clip(1, 45)
pages_visited = np.random.poisson(4, n_samples) + 1
previous_purchases = np.random.choice([0,1,2,3,4,5], n_samples, p=[0.45,0.25,0.15,0.08,0.04,0.03])
log_odds = -3.5 + 0.12*time_spent + 0.35*pages_visited + 0.60*previous_purchases
probs = 1/(1+np.exp(-log_odds))
y = np.random.binomial(1, probs)
X = np.column_stack([time_spent, pages_visited, previous_purchases])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7, None],
    'max_features': ['sqrt', 'log2']
}

start = time.time()
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)
elapsed = time.time() - start

print(f"Total combinations tried: {len(param_grid['n_estimators'])*len(param_grid['max_depth'])*len(param_grid['max_features'])}")
print(f"Time taken: {elapsed:.2f}s")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.2%}")

final_model = grid_search.best_estimator_
test_acc = final_model.score(X_test, y_test)
print(f"Final Test Accuracy: {test_acc:.2%}")

Total combinations tried: 24
Time taken: 12.15s
Best params: {'max_depth': 5, 'max_features': 'sqrt', 'n_estimators': 50}
Best CV score: 77.50%
Final Test Accuracy: 80.00%


#### RandomSearch CV

In [6]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'n_estimators': [50, 100, 150, 200, 250],
    'max_depth': [3, 5, 7, 10, None],
    'max_features': ['sqrt', 'log2']
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), 
    param_distributions, 
    n_iter=10,          # try only 10 random combinations (instead of all 50)
    cv=5, 
    random_state=42
)

#### RandomSearch vs GridSearch

In [7]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import time

np.random.seed(42)
n_samples = 300
time_spent = np.random.normal(12, 5, n_samples).clip(1, 45)
pages_visited = np.random.poisson(4, n_samples) + 1
previous_purchases = np.random.choice([0,1,2,3,4,5], n_samples, p=[0.45,0.25,0.15,0.08,0.04,0.03])
log_odds = -3.5 + 0.12*time_spent + 0.35*pages_visited + 0.60*previous_purchases
probs = 1/(1+np.exp(-log_odds))
y = np.random.binomial(1, probs)
X = np.column_stack([time_spent, pages_visited, previous_purchases])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bigger grid this time - more realistic scenario
param_grid = {
    'n_estimators': [50, 100, 150, 200, 250],
    'max_depth': [3, 5, 7, 10, None],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10]
}
total_combos = 5*5*2*3
print(f"Total possible combinations: {total_combos}")

# Grid Search - ALL combinations
start = time.time()
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)
grid_time = time.time() - start
print(f"\nGrid Search: tried ALL {total_combos} combos, took {grid_time:.1f}s")
print(f"Best CV score: {grid.best_score_:.2%}, params: {grid.best_params_}")

# Random Search - only 15 random combinations
start = time.time()
random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_grid, n_iter=15, cv=5, random_state=42, scoring='accuracy')
random_search.fit(X_train, y_train)
random_time = time.time() - start
print(f"\nRandom Search: tried only 15 combos, took {random_time:.1f}s")
print(f"Best CV score: {random_search.best_score_:.2%}, params: {random_search.best_params_}")

print(f"\nSpeed-up: Random Search was {grid_time/random_time:.1f}x faster")
print(f"Score difference: {abs(grid.best_score_ - random_search.best_score_)*100:.2f} percentage points")

Total possible combinations: 150

Grid Search: tried ALL 150 combos, took 240.4s
Best CV score: 79.17%, params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 200}

Random Search: tried only 15 combos, took 52.9s
Best CV score: 77.50%, params: {'n_estimators': 250, 'min_samples_split': 2, 'max_features': 'log2', 'max_depth': 3}

Speed-up: Random Search was 4.5x faster
Score difference: 1.67 percentage points


In [8]:
pip install optuna


   ---------- ----------------------------- 1/4 [colorlog]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ---------------------------------------- 4/4 [optuna]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Bayesian Optimization

In [9]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=5).mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)   # only 30 "smart" trials

print(f"Best params: {study.best_params}")
print(f"Best score: {study.best_value:.2%}")

c:\Users\Arshad\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-09-13 10:03:25,465] A new study created in memory with name: no-name-54af8550-0ca4-4c5f-b387-9c9ba4633fc9
[I 2026-09-13 10:03:25,970] Trial 0 finished with value: 0.7166666666666667 and parameters: {'n_estimators': 80, 'max_depth': 19}. Best is trial 0 with value: 0.7166666666666667.
[I 2026-09-13 10:03:26,474] Trial 1 finished with value: 0.7666666666666666 and parameters: {'n_estimators': 109, 'max_depth': 4}. Best is trial 1 with value: 0.7666666666666666.
[I 2026-09-13 10:03:27,192] Trial 2 finished with value: 0.7666666666666666 and parameters: {'n_estimators': 154, 'max_depth': 8}. Best is trial 1 with value: 0.7666666666666666.
[I 2026-09-13 10:03:27,869] Trial 3 finished with value: 0.72083333333

Best params: {'n_estimators': 195, 'max_depth': 6}
Best score: 77.08%
